# <font color="#418FDE" size="6.5" uppercase>**Logistic LDA QDA**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Fit logistic regression models for binary and multiclass classification. 
- Configure penalties, solvers, and class weights for practical classification problems. 
- Compare Logistic Regression, LDA, and QDA on suitable datasets. 


## **1. Logistic Regression Models**

### **1.1. Binary Logistic Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_01.jpg?v=1783995819" width="250">



>* Classifies outcomes with two possible categories
>* Converts feature scores into thresholded probabilities

>* Features shift positive-class probability up or down
>* Probabilities support interpretation and nuanced decisions

>* Train on labeled data, evaluate with metrics
>* Tune thresholds and check model assumptions



In [ ]:
#@title Python Code - Binary Logistic Regression

# Fit binary logistic regression on cancer data.
# Predict probabilities before choosing class labels.
# Compare accuracy and positive class probabilities.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
cancer = load_breast_cancer()
features = cancer.data
target = cancer.target

# Validate the basic binary classification shape.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

if len(np.unique(target)) != 2:
    raise ValueError("This example needs exactly two classes.")

# Split data while preserving the class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.25, stratify=target, random_state=42
)

# Scale features, then fit one logistic regression model.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Predict both probabilities and final class labels.
positive_probabilities = model.predict_proba(X_test)[:, 1]
predicted_labels = model.predict(X_test)
accuracy = accuracy_score(y_test, predicted_labels)

# Show concise results that connect probabilities to labels.
print("scikit-learn version:", sklearn.__version__)
print("Positive class:", cancer.target_names[1])
print("Test accuracy:", round(accuracy, 3))
print("First five positive-class probabilities:")
print(np.round(positive_probabilities[:5], 3))
print("First five predicted labels:")
print(predicted_labels[:5])



### **1.2. Multiclass Logistic**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_02.jpg?v=1783995823" width="250">



>* Handles classification with three or more classes
>* Outputs class probabilities showing prediction confidence

>* One-vs-rest trains separate binary classifiers
>* Multinomial models compare all classes jointly

>* Linear feature effects shape class probabilities
>* Useful baseline, but validate probabilities carefully



In [ ]:
#@title Python Code - Multiclass Logistic

# This example fits multiclass logistic regression.
# It uses iris flower measurements and labels.
# The output shows probabilities across three classes.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged dataset with three classes.
iris = load_iris()
X = iris.data
y = iris.target

# Check that the dataset matches the multiclass lesson.
if X.shape[0] != y.shape[0] or len(np.unique(y)) != 3:
    raise ValueError("Expected matching rows and three classes.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

# Scale features, then fit one multinomial logistic model.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=300, random_state=42)
)

model.fit(X_train, y_train)

# Predict labels and class probabilities on unseen examples.
y_pred = model.predict(X_test)
probabilities = model.predict_proba(X_test[:3])

accuracy = accuracy_score(y_test, y_pred)
class_names = iris.target_names

print("scikit-learn version:", sklearn.__version__)
print("Test accuracy:", round(accuracy, 3))
print("Classes:", ", ".join(class_names))

# Show how each row receives probabilities summing to one.
for row_number in range(3):
    rounded = np.round(probabilities[row_number], 3)
    predicted_name = class_names[np.argmax(probabilities[row_number])]
    print("Example", row_number + 1, "probabilities:", rounded, "->", predicted_name)

print("Each probability row sums to:", np.round(probabilities.sum(axis=1), 3))



### **1.3. Logistic Penalties**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_01_03.jpg?v=1783995821" width="250">



>* Penalties limit overly flexible logistic models
>* Smaller weights improve stability and generalization

>* Smooth penalties shrink all coefficients gently
>* Sparse penalties remove weak predictors

>* Tune penalty strength with validation or cross-validation
>* Control multiclass coefficients for stable models



In [ ]:
#@title Python Code - Logistic Penalties

# Compare logistic penalties on one classification task.
# Penalties shrink coefficients to reduce overfitting.
# Results show accuracy and coefficient size.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small built-in binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Check that features and labels match correctly.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match label count.")

# Split data before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# Fit the same model with three penalty choices.
penalty_settings = [("none", None), ("l2", "l2"), ("l1", "l1")]
results = []

for label, penalty in penalty_settings:
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            penalty=penalty, solver="saga", C=0.2, max_iter=3000, random_state=42
        ),
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    coefficients = model.named_steps["logisticregression"].coef_[0]

    accuracy = accuracy_score(y_test, predictions)
    average_size = abs(coefficients).mean()
    zero_count = int((abs(coefficients) < 0.001).sum())
    results.append((label, accuracy, average_size, zero_count))

print("scikit-learn version:", sklearn.__version__)
print("Penalty | test accuracy | mean |coef| | near-zero coefficients")

for label, accuracy, average_size, zero_count in results:
    print(label, "|", round(accuracy, 3), "|", round(average_size, 3), "|", zero_count)



## **2. Solver Choices**

### **2.1. Solver Compatibility**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_01.jpg?v=1783995831" width="250">



>* Solvers must match data and classification needs
>* Penalties require compatible optimization methods

>* Penalties need compatible solvers
>* Choose regularization to match modeling goals

>* Match solvers to multiclass strategy
>* Check scaling, memory, and convergence needs



In [ ]:
#@title Python Code - Solver Compatibility

# This script checks logistic regression solver compatibility.
# Penalties must match solvers before model fitting.
# The output shows valid and invalid choices.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small packaged binary classification dataset.
data = load_breast_cancer()
X = data.data
y = data.target

# Validate the basic shape before modeling.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data with stratification to preserve class balance.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

# These settings demonstrate common solver and penalty pairings.
settings = [
    ("lbfgs + l2", "lbfgs", "l2", None),
    ("liblinear + l1", "liblinear", "l1", None),
    ("saga + elasticnet", "saga", "elasticnet", 0.5),
    ("lbfgs + l1", "lbfgs", "l1", None),
]

print("scikit-learn version:", sklearn.__version__)
print("Solver compatibility check on breast cancer data:")

# Fit each configuration and report whether it is supported.
for label, solver, penalty, l1_ratio in settings:
    model = LogisticRegression(
        solver=solver, penalty=penalty, l1_ratio=l1_ratio, max_iter=2000
    )
    pipeline = make_pipeline(StandardScaler(), model)
    try:
        pipeline.fit(X_train, y_train)
        accuracy = pipeline.score(X_test, y_test)
        print(label, "works, test accuracy =", round(accuracy, 3))
    except ValueError:
        print(label, "fails because this solver cannot use that penalty")

# The final line confirms the main lesson.
print("Choose the solver and penalty as a compatible pair.")



### **2.2. Class Weighting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_02.jpg?v=1783995833" width="250">



>* Imbalanced data can bias models toward majorities
>* Class weights emphasize rarer or important classes

>* Weights reshape loss and decision boundaries
>* Choose weights by real error costs

>* Balanced weights adjust for class frequency
>* Validate weights against real error costs



In [ ]:
#@title Python Code - Class Weighting

# This example compares logistic class weighting.
# Balanced weights emphasize the minority class.
# Recall usually improves while precision may change.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small imbalanced binary classification dataset.
features, target = make_classification(
    n_samples=1200,
    n_features=6,
    n_informative=4,
    n_redundant=0,
    weights=[0.9, 0.1],
    class_sep=0.9,
    random_state=42,
)

# Check that both classes are present.
class_counts = np.bincount(target)
if len(class_counts) != 2:
    raise ValueError("Expected exactly two classes.")

# Split data while preserving the class imbalance.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Fit scaling only on training data.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the same model with and without class weighting.
plain_model = LogisticRegression(solver="lbfgs", max_iter=500, random_state=42)
weighted_model = LogisticRegression(
    solver="lbfgs",
    class_weight="balanced",
    max_iter=500,
    random_state=42,
)

plain_model.fit(X_train_scaled, y_train)
weighted_model.fit(X_train_scaled, y_train)

# Compare minority-class precision and recall.
plain_predictions = plain_model.predict(X_test_scaled)
weighted_predictions = weighted_model.predict(X_test_scaled)

plain_recall = recall_score(y_test, plain_predictions)
weighted_recall = recall_score(y_test, weighted_predictions)
plain_precision = precision_score(y_test, plain_predictions)
weighted_precision = precision_score(y_test, weighted_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Training class counts: majority=756, minority=84")
print("No weights: precision=", round(plain_precision, 3), "recall=", round(plain_recall, 3))
print("Balanced weights: precision=", round(weighted_precision, 3), "recall=", round(weighted_recall, 3))
print("Class weighting changes the error trade-off.")



### **2.3. Sample Weighting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_02_03.jpg?v=1783995835" width="250">



>* Weights control each example’s model influence
>* Useful for reliability, recency, or importance

>* Correct unrepresentative training data with weights
>* Match model influence to real-world importance

>* Extreme weights can destabilize model fitting
>* Document weights and validate regularization choices



In [ ]:
#@title Python Code - Sample Weighting

# This example shows individual sample weights.
# Weights change logistic regression training influence.
# The printed results compare weighted and unweighted models.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Create a small imbalanced binary classification dataset.
features, target = make_classification(
    n_samples=600,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    weights=[0.9, 0.1],
    class_sep=0.8,
    random_state=42,
)

# Split before scaling to avoid test data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.3,
    stratify=target,
    random_state=42,
)

# Validate that both classes are present after splitting.
if len(np.unique(y_train)) != 2 or len(np.unique(y_test)) != 2:
    raise ValueError("Both splits must contain both classes.")

# Fit scaling only on the training features.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Give positive training examples more individual influence.
sample_weights = np.ones(len(y_train))
sample_weights[y_train == 1] = 8.0

# Train the same model once without weights.
plain_model = LogisticRegression(solver="lbfgs", max_iter=300, random_state=42)
plain_model.fit(X_train_scaled, y_train)

# Train the same model again with sample weights.
weighted_model = LogisticRegression(solver="lbfgs", max_iter=300, random_state=42)
weighted_model.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# Compare how many positive cases each model catches.
plain_predictions = plain_model.predict(X_test_scaled)
weighted_predictions = weighted_model.predict(X_test_scaled)

plain_recall = recall_score(y_test, plain_predictions)
weighted_recall = recall_score(y_test, weighted_predictions)

print("scikit-learn version:", sklearn.__version__)
print("Positive class share in training:", round(float(np.mean(y_train)), 3))
print("Unweighted positive recall:", round(float(plain_recall), 3))
print("Sample-weighted positive recall:", round(float(weighted_recall), 3))
print("Positive sample weight used:", round(float(sample_weights.max()), 1))



## **3. Discriminant Analysis**

### **3.1. LDA Classifier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_01.jpg?v=1783995825" width="250">



>* Models class distributions probabilistically
>* Creates linear boundaries with shared covariance

>* LDA and logistic regression differ conceptually
>* LDA handles multiclass compact clusters well

>* Use LDA for similar elliptical class clusters
>* Compare with QDA and logistic regression



In [ ]:
#@title Python Code - LDA Classifier

# This example trains an LDA classifier.
# It compares LDA with logistic regression.
# The plot shows their linear boundaries.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Use two iris classes and two measurable features.
iris = load_iris()
X = iris.data[:100, [2, 3]]
y = iris.target[:100]

# Check that the small teaching dataset has matching rows.
if X.shape[0] != y.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data so evaluation uses unseen flowers.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Fit LDA and logistic regression on the same training data.
lda = LinearDiscriminantAnalysis()
logistic = LogisticRegression(random_state=42, max_iter=200)
lda.fit(X_train, y_train)
logistic.fit(X_train, y_train)

# Compare test accuracy for the two linear classifiers.
lda_accuracy = accuracy_score(y_test, lda.predict(X_test))
logistic_accuracy = accuracy_score(y_test, logistic.predict(X_test))
print("scikit-learn version:", sklearn.__version__)
print("LDA test accuracy:", round(lda_accuracy, 3))
print("Logistic regression test accuracy:", round(logistic_accuracy, 3))

# Build a grid to visualize each model boundary.
x_min = X[:, 0].min() - 0.3
x_max = X[:, 0].max() + 0.3
y_min = X[:, 1].min() - 0.2
y_max = X[:, 1].max() + 0.2

# Predict class probabilities across the feature space.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]
lda_scores = lda.predict_proba(grid)[:, 1].reshape(xx.shape)
logistic_scores = logistic.predict_proba(grid)[:, 1].reshape(xx.shape)

# Plot points and the two learned decision boundaries.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k")
ax.contour(xx, yy, lda_scores, levels=[0.5], colors="black", linewidths=2)
ax.contour(xx, yy, logistic_scores, levels=[0.5], colors="green", linestyles="--")

# Label the plot for beginner interpretation.
ax.set_title("LDA and Logistic Regression Boundaries")
ax.set_xlabel("Petal length in centimeters")
ax.set_ylabel("Petal width in centimeters")
ax.legend(scatter.legend_elements()[0], ["setosa", "versicolor"], title="Class")
plt.show()



### **3.2. QDA Classifier**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_02.jpg?v=1783995829" width="250">



>* QDA models each class’s unique variability
>* Curved boundaries handle non-linear class separation

>* QDA captures curved, class-specific patterns
>* More flexibility requires more reliable data

>* Use QDA with enough class data
>* Validate against simpler, more stable models



In [ ]:
#@title Python Code - QDA Classifier

# This example compares three classification boundaries.
# QDA can learn curved class separation.
# The plot shows where QDA helps.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_gaussian_quantiles
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# Generate two classes with a curved circular pattern.
features, target = make_gaussian_quantiles(
    n_samples=600, n_features=2, n_classes=2, random_state=42
)

# Check that the generated data has the expected shape.
if features.shape != (600, 2):
    raise ValueError("Expected 600 rows and 2 features.")

# Split data so evaluation uses unseen examples.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Fit three classifiers for the same classification task.
models = {
    "Logistic": LogisticRegression(max_iter=500, random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "QDA": QuadraticDiscriminantAnalysis(),
}

# Train each model and store its test accuracy.
accuracies = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracies[name] = accuracy_score(y_test, predictions)

# Print concise results before showing the boundary.
print(f"scikit-learn version: {sklearn.__version__}")
print(f"Logistic accuracy: {accuracies['Logistic']:.3f}")
print(f"LDA accuracy: {accuracies['LDA']:.3f}")
print(f"QDA accuracy: {accuracies['QDA']:.3f}")

# Build a grid to visualize QDA predictions.
x_min = features[:, 0].min() - 0.5
x_max = features[:, 0].max() + 0.5
y_min = features[:, 1].min() - 0.5
y_max = features[:, 1].max() + 0.5

# Predict QDA classes across the feature space.
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 180), np.linspace(y_min, y_max, 180)
)
grid_points = np.c_[xx.ravel(), yy.ravel()]
qda_grid = models["QDA"].predict(grid_points).reshape(xx.shape)

# Plot the curved QDA decision regions and data.
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, qda_grid, alpha=0.25, cmap="coolwarm")
scatter = ax.scatter(
    features[:, 0], features[:, 1], c=target, cmap="coolwarm", s=18
)

# Label the plot for beginner interpretation.
ax.set_title("QDA learns a curved boundary on circular data")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(*scatter.legend_elements(), title="Class")
plt.show()



### **3.3. Covariance Shrinkage**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_11/Lecture_A/image_03_03.jpg?v=1783995827" width="250">



>* Stabilizes covariance estimates in difficult datasets
>* Improves generalization by reducing sampling noise

>* Logistic, LDA, and QDA differ in assumptions
>* Shrinkage reduces overfitting in covariance estimates

>* Best for correlated, high-dimensional datasets
>* Validate carefully for reliable generalization



In [ ]:
#@title Python Code - Covariance Shrinkage

# Compare discriminant models with covariance shrinkage.
# Shrinkage stabilizes covariance estimates in small samples.
# Test accuracy shows the practical tradeoff.

import numpy as np
import sklearn
from sklearn.datasets import make_classification
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small high-dimensional classification dataset.
features, target = make_classification(
    n_samples=160,
    n_features=20,
    n_informative=6,
    n_redundant=8,
    n_classes=2,
    class_sep=1.0,
    random_state=42,
)

# Check that the example has more features than classes.
if features.shape[1] < 10:
    raise ValueError("Expected a high-dimensional teaching dataset.")

# Split data before scaling to avoid data leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Define comparable classifiers for the same train-test split.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "LDA no shrinkage": LinearDiscriminantAnalysis(solver="lsqr"),
    "LDA shrinkage": LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto"),
    "QDA regularized": QuadraticDiscriminantAnalysis(reg_param=0.4),
}

# Fit each model after standardizing the training data only.
results = []
for model_name, model in models.items():
    pipeline = make_pipeline(StandardScaler(), model)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append((model_name, accuracy))

# Print concise results for comparing the classifiers.
print("scikit-learn version:", sklearn.__version__)
print("Dataset shape:", features.shape[0], "rows and", features.shape[1], "features")
for model_name, accuracy in results:
    print(model_name + ":", round(accuracy, 3))



# <font color="#418FDE" size="6.5" uppercase>**Logistic LDA QDA**</font>


In this lecture, you learned to:
- Fit logistic regression models for binary and multiclass classification. 
- Configure penalties, solvers, and class weights for practical classification problems. 
- Compare Logistic Regression, LDA, and QDA on suitable datasets. 

In the next Lecture (Lecture B), we will go over 'Stochastic Gradient'